In [1]:
!pip install transformers torch sentencepiece fasttext-wheel


  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     ----------------------------- ---------- 30.7/41.5 kB 1.4 MB/s eta 0:00:01
     -------------------------------------- 41.5/41.5 kB 504.6 kB/s eta 0:00:00
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached pybind11-3.0.1-py3-none-any.whl.metadata (10.0 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
Using cached transformers-4.57.3-py3-none-any.


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from transformers import pipeline, M2M100ForConditionalGeneration, M2M100Tokenizer
import fasttext
from transformers import PegasusTokenizer

pegasus_tokenizer = PegasusTokenizer.from_pretrained("google/pegasus-xsum")


In [6]:
lang_model = fasttext.load_model("lid.176.bin")

def detect_language(text):
    prediction = lang_model.predict(text.replace("\n", " "), k=1)
    return prediction[0][0].replace("__label__", "")


In [7]:
tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")
model = M2M100ForConditionalGeneration.from_pretrained(
    "facebook/m2m100_418M"
)

def translate_to_english(text, src_lang):
    tokenizer.src_lang = src_lang
    encoded = tokenizer(text, return_tensors="pt", truncation=True)

    generated = model.generate(
        **encoded,
        forced_bos_token_id=tokenizer.get_lang_id("en")
    )

    return tokenizer.decode(generated[0], skip_special_tokens=True)



In [8]:
summarizer = pipeline(
    "summarization",
    model="google/pegasus-xsum"
)

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cpu


In [14]:
def summarize_english(text):
    return summarizer(
        text,
        max_length=130,
        min_length=40,
        do_sample=False,
        truncation=True
    )[0]["summary_text"]

In [11]:
def process_article(text):
    lang = detect_language(text)

    if lang != "en":
        text = translate_to_english(text, lang)

    summary = summarize_english(text)

    return {
        "detected_language": lang,
        "summary": summary
    }


In [15]:
article = """Daryl Mitchell stood tall with a stroke-filled half-century to help New Zealand post 300 for 8 in the first One-Day International against India in Vadodara on Sunday (January 11, 2026). Mitchell, ranked world No. 3 behind India superstars Virat Kohli and Rohit Sharma, lived up to expectations with a gritty knock which gave New Zealand a much-needed impetus amid middle order collapse. Mitchell clobbered five fours and three sixes to make 84 off 71 balls after New Zealand middle order failed to build on an ideal start given by Devon Conway (56 off 67 balls; 6 fours, 1 six) and Henry Nicholls (62 off 69 balls; 8 fours). Openers Conway and Nicholls had put on 117 runs to defy India for the first 21 overs, but Harshit Rana’s (2/65) second spell of 2-0-13-2 turned the tables and the home side chipped away with regular strikes. New Zealand skidded from 117 for no loss in the 22nd over to 198 for five in the 38th. But Mitchell forged vital stands to come to their rescue, while debutant Kristian Clarke struck three late fours to make 17-ball 24 not out. In the 22nd over, Rana took pace off the ball to break the century partnership between Conway and Nicholls. An innocuous off-cutter, bowled away from the batter saw Nicholls reaching out for it, only to get an edge that carried to the wicketkeeper for India’s first breakthrough. In the 24th over, Rana mixed up the slower ones with a few in excess of 140 kmph to keep the Kiwis guessing. On the final delivery of the over, he had Conway’s inside edge crashing into the wickets. But India and Rana would not have had to wait for a breakthrough had Kuldeep Yadav (1/52) hung on to a regulation catch at third man in the fifth over, when Nicholls was on five. Siraj sent down a slower one which had Will Young feathering one behind the wickets, while Prasidh Krishna (2/60) cleaned up Mitchell Hay (12) in the 37th. Under the weather for the last few days, Glenn Phillips appeared a tad sluggish in his brief stay which was ended by Kuldeep, and a brilliant direct hit by Shreyas Iyer from long-on caught Kiwi skipper Michael Bracewell short of his crease. Earlier, Conway began with a drive down the wicket for his first boundary while Nicholls took his time to get going, and used the sweep shot well at times to find the odd boundary. The duo grew in confidence to bring out the sweep and reverse sweeps with perfection while rotating the strike well to put on the century stand."""
process_article(article)


{'detected_language': 'en',
 'summary': 'All photographs  BCCI.com /Suresh K Choudhary / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum Photos / Magnum'}